In [2]:
import pandas as pd
import os

# Load the rationales data
data_path = "../../crest/data/rationales/my_movies_test_sparsemap_50p.tsv"
df = pd.read_csv(data_path, sep='\t')

print(f"Data loaded successfully!")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print("\nFirst few rows:")
df.head()

Data loaded successfully!
Shape: (199, 4)
Columns: ['texts', 'labels', 'predictions', 'z']

First few rows:


,texts,labels,predictions,z
0,▁there ▁may ▁not ▁be ▁ a ▁critic ▁alive ▁who ▁...,0,0,"[0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, ..."
1,▁ ren e e ▁ zel l weg er ▁stars ▁as ▁ s onia ▁...,0,0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, ..."
2,▁there ▁ ' re ▁so ▁many ▁things ▁to ▁critic iz...,0,0,"[0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 1.0, ..."
3,▁do ▁ n ' t ▁let ▁this ▁movie ▁fool ▁you ▁into...,0,0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, ..."
4,▁it ▁ ' s ▁ a ▁good ▁thing ▁most ▁animated ▁sc...,0,0,"[0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, ..."


In [9]:
import numpy as np
import ast

def process_text_and_rationales(text, z_values):
    """
    Convert tokenized text to normal form and aggregate rationales at word level.
    
    Args:
        text (str): Tokenized text with ▁ markers
        z_values (list): Binary rationale values for each token
    
    Returns:
        tuple: (normal_text, word_rationales, rationale_ranges)
    """
    # Split text into tokens
    tokens = text.split()
    
    # Parse z_values if it's a string representation of a list
    if isinstance(z_values, str):
        z_values = ast.literal_eval(z_values)
    
    words = []
    word_rationales = []
    current_word = ""
    current_word_has_rationale = False
    
    for i, token in enumerate(tokens):
        z_val = z_values[i] if i < len(z_values) else 0.0
        
        if token.startswith('▁'):
            # Start of a new word
            if current_word:
                # Save the previous word
                words.append(current_word)
                word_rationales.append(1.0 if current_word_has_rationale else 0.0)
            
            # Start new word (remove ▁ marker)
            current_word = token[1:]  # Remove ▁
            current_word_has_rationale = (z_val == 1.0)
        else:
            # Continuation of current word
            current_word += token
            if z_val == 1.0:
                current_word_has_rationale = True
    
    # Don't forget the last word
    if current_word:
        words.append(current_word)
        word_rationales.append(1.0 if current_word_has_rationale else 0.0)
    
    # Join words with spaces to form normal text
    normal_text = ' '.join(words)
    
    # Find rationale ranges
    rationale_ranges = []
    start_idx = None
    
    for i, rationale in enumerate(word_rationales):
        if rationale == 1.0:
            if start_idx is None:
                start_idx = i
        else:
            if start_idx is not None:
                rationale_ranges.append([start_idx, i])
                start_idx = None
    
    # Don't forget if the last sequence ends with rationale
    if start_idx is not None:
        rationale_ranges.append([start_idx, len(word_rationales)])
    
    return normal_text, word_rationales, rationale_ranges

# Apply the processing to all rows
df['normal_text'] = ''
df['word_rationales'] = ''
df['rationale_ranges'] = ''

for idx, row in df.iterrows():
    normal_text, word_rationales, rationale_ranges = process_text_and_rationales(row['texts'], row['z'])
    df.at[idx, 'normal_text'] = normal_text
    df.at[idx, 'word_rationales'] = word_rationales
    df.at[idx, 'rationale_ranges'] = rationale_ranges

print("Text processing completed!")
print(f"Sample processed text:\n{df.iloc[0]['normal_text']}")
print(f"Original tokens: {len(df.iloc[0]['texts'].split())}")
print(f"Words after processing: {len(df.iloc[0]['normal_text'].split())}")
print(f"Word rationales: {df.iloc[0]['word_rationales'][:10]}...")  # Show first 10
print(f"Rationale ranges: {df.iloc[0]['rationale_ranges'][:5]}...")  # Show first 5 ranges

Text processing completed!
Sample processed text:
there may not be a critic alive who harbors as much affection for shlock monster movies as i do . i delighted in the sneaky - smart entertainment of ron underwood 's big - underground - worm yarn tremors ; i even giggled at last year 's critically - savaged big - underwater - snake yarn anaconda . something about these films causes me to lower my inhibitions and return to the saturday afternoons of my youth , spent in the company of ghidrah , the creature from the black lagoon and the blob . deep rising , a big - undersea - serpent yarn , does n't quite pass the test . sure enough , all the modern monster movie ingredients are in place : a conspicuously multi - ethnic / multi - national collection of bait . .. excuse me , characters ; an isolated location , here a derelict cruise ship in the south china sea ; some comic relief ; a few cgi - enhanced gross - outs ; and at least one big explosion . there are too - cheesy - to - be - accid

In [ ]:
# Let's examine a few examples to see the transformation
def display_text_comparison(idx):
    """Display original vs processed text with rationales"""
    row = df.iloc[idx]
    
    print(f"Example {idx + 1}:")
    print(f"Label: {row['labels']} (0=negative, 1=positive)")
    print(f"Prediction: {row['predictions']}")
    print()
    
    print("Original tokenized text:")
    print(row['texts'][:200] + "..." if len(row['texts']) > 200 else row['texts'])
    print()
    
    print("Processed normal text:")
    print(row['normal_text'][:200] + "..." if len(row['normal_text']) > 200 else row['normal_text'])
    print()
    
    # Show words with their rationale values
    words = row['normal_text'].split()
    rationales = row['word_rationales']
    
    print("Words with rationales (showing first 20 words):")
    for i in range(min(20, len(words))):
        word = words[i]
        rat_val = rationales[i] if i < len(rationales) else 0.0
        marker = "**" if rat_val == 1.0 else ""
        print(f"{marker}{word}{marker}", end=" ")
    print("\n" + "="*80)

# Display first 3 examples
for i in range(3):
    display_text_comparison(i)

Example 1:
Label: 0 (0=negative, 1=positive)
Prediction: 0

Original tokenized text:
▁there ▁may ▁not ▁be ▁ a ▁critic ▁alive ▁who ▁harbor s ▁as ▁much ▁affection ▁for ▁sh lock ▁monster ▁movies ▁as ▁ i ▁do ▁ . ▁ i ▁delighted ▁in ▁the ▁sneak y ▁ - ▁smart ▁entertainment ▁of ▁ r on ▁under ...

Processed normal text:
there may not be a critic alive who harbors as much affection for shlock monster movies as i do . i delighted in the sneaky - smart entertainment of ron underwood 's big - underground - worm yarn trem...

Words with rationales (showing first 20 words):
there **may** **not** be a **critic** **alive** **who** **harbors** **as** **much** **affection** for **shlock** **monster** **movies** **as** i **do** . 
Example 2:
Label: 0 (0=negative, 1=positive)
Prediction: 0

Original tokenized text:
▁ ren e e ▁ zel l weg er ▁stars ▁as ▁ s onia ▁ , ▁ a ▁young ▁je w ish ▁wife ▁and ▁mother ▁frustrated ▁by ▁the ▁constraints ▁of ▁her ▁has i dic ▁community ▁in ▁ brook ly n ▁ . ▁her ▁husband ▁( ▁ 

In [10]:
# Demonstrate rationale ranges functionality
def display_rationale_ranges(idx):
    """Display text with highlighted rationale ranges"""
    row = df.iloc[idx]
    words = row['normal_text'].split()
    ranges = row['rationale_ranges']
    
    print(f"Example {idx + 1} - Rationale Ranges:")
    print(f"Label: {row['labels']} (0=negative, 1=positive)")
    print(f"Prediction: {row['predictions']}")
    print(f"Total words: {len(words)}")
    print(f"Rationale ranges: {ranges}")
    print()
    
    # Create a highlighted version of the text
    highlighted_words = words.copy()
    for start, end in ranges:
        for i in range(start, min(end, len(highlighted_words))):
            highlighted_words[i] = f"**{highlighted_words[i]}**"
    
    print("Text with highlighted rationale ranges:")
    print(" ".join(highlighted_words[:50]) + ("..." if len(highlighted_words) > 50 else ""))
    print()
    
    # Show specific ranges
    print("Rationale ranges details:")
    for i, (start, end) in enumerate(ranges[:5]):  # Show first 5 ranges
        range_words = words[start:end]
        print(f"  Range {i+1}: [{start}, {end}] -> '{' '.join(range_words)}'")
    
    print("="*80)

# Display examples
for i in range(2):
    display_rationale_ranges(i)

Example 1 - Rationale Ranges:
Label: 0 (0=negative, 1=positive)
Prediction: 0
Total words: 323
Rationale ranges: [[1, 3], [5, 12], [13, 17], [18, 19], [21, 28], [32, 33], [34, 35], [38, 39], [41, 44], [47, 48], [50, 51], [58, 66], [67, 74], [77, 79], [80, 82], [85, 87], [89, 90], [92, 94], [95, 97], [99, 100], [105, 110], [113, 115], [116, 123], [127, 129], [131, 133], [134, 136], [137, 138], [140, 141], [143, 147], [153, 154], [158, 164], [165, 166], [168, 170], [177, 179], [182, 183], [184, 185], [197, 198], [200, 201], [203, 205], [224, 228], [230, 231], [236, 247], [248, 252], [285, 287], [294, 298], [305, 310], [311, 313], [315, 319], [321, 323]]

Text with highlighted rationale ranges:
there **may** **not** be a **critic** **alive** **who** **harbors** **as** **much** **affection** for **shlock** **monster** **movies** **as** i **do** . i **delighted** **in** **the** **sneaky** **-** **smart** **entertainment** of ron underwood 's **big** - **underground** - worm yarn **tremors**

In [11]:
# Summary statistics
print("Data Processing Summary:")
print(f"Total reviews: {len(df)}")
print(f"Average original tokens per review: {df['texts'].apply(lambda x: len(x.split())).mean():.1f}")
print(f"Average words per review after processing: {df['normal_text'].apply(lambda x: len(x.split())).mean():.1f}")

# Calculate rationale statistics
avg_rationale_percentage = []
avg_ranges_per_review = []
avg_range_lengths = []

for idx, row in df.iterrows():
    rationales = row['word_rationales']
    ranges = row['rationale_ranges']
    
    if len(rationales) > 0:
        percentage = (sum(rationales) / len(rationales)) * 100
        avg_rationale_percentage.append(percentage)
    
    avg_ranges_per_review.append(len(ranges))
    
    if ranges:
        range_lengths = [end - start for start, end in ranges]
        avg_range_lengths.extend(range_lengths)

print(f"Average percentage of words marked as important: {np.mean(avg_rationale_percentage):.1f}%")
print(f"Average number of rationale ranges per review: {np.mean(avg_ranges_per_review):.1f}")
print(f"Average length of rationale ranges: {np.mean(avg_range_lengths):.1f} words")

# Display the new dataframe structure
print(f"\nNew DataFrame columns: {list(df.columns)}")
print(f"DataFrame shape: {df.shape}")

# Show a brief sample
print(f"\nSample of processed data:")
df[['labels', 'predictions', 'normal_text', 'word_rationales', 'rationale_ranges']].head(2)

Data Processing Summary:
Total reviews: 199
Average original tokens per review: 505.8
Average words per review after processing: 353.6
Average percentage of words marked as important: 40.0%
Average number of rationale ranges per review: 57.2
Average length of rationale ranges: 2.5 words

New DataFrame columns: ['texts', 'labels', 'predictions', 'z', 'normal_text', 'word_rationales', 'rationale_ranges']
DataFrame shape: (199, 7)

Sample of processed data:


,labels,predictions,normal_text,word_rationales,rationale_ranges
0,0,0,there may not be a critic alive who harbors as...,"[0.0, 1.0, 1.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, ...","[[1, 3], [5, 12], [13, 17], [18, 19], [21, 28]..."
1,0,0,"renee zellweger stars as sonia , a young jewis...","[0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, ...","[[1, 3], [8, 9], [10, 13], [19, 20], [21, 22],..."


In [3]:
# Analysis of reviews with exactly 512 original tokens
token_counts = df['texts'].apply(lambda x: len(x.split()))

# Count reviews with exactly 512 tokens
reviews_with_512_tokens = (token_counts == 512).sum()
total_reviews = len(df)
percentage_512_tokens = (reviews_with_512_tokens / total_reviews) * 100

print(f"Reviews with exactly 512 original tokens: {reviews_with_512_tokens}")
print(f"Total reviews: {total_reviews}")
print(f"Percentage of reviews with 512 tokens: {percentage_512_tokens:.2f}%")

# Additional statistics about token distribution
print(f"\nToken count statistics:")
print(f"Minimum tokens: {token_counts.min()}")
print(f"Maximum tokens: {token_counts.max()}")
print(f"Mean tokens: {token_counts.mean():.1f}")
print(f"Median tokens: {token_counts.median():.1f}")

# Show distribution around 512
print(f"\nDistribution around 512 tokens:")
for target in [510, 511, 512, 513, 514]:
    count = (token_counts == target).sum()
    pct = (count / total_reviews) * 100
    print(f"  {target} tokens: {count} reviews ({pct:.2f}%)")

# Check if there's a pattern (truncation at 512?)
print(f"\nReviews with 512 or more tokens: {(token_counts >= 512).sum()} ({((token_counts >= 512).sum() / total_reviews) * 100:.2f}%)")
print(f"Reviews with exactly 512 tokens: {reviews_with_512_tokens} ({percentage_512_tokens:.2f}%)")

Reviews with exactly 512 original tokens: 187
Total reviews: 199
Percentage of reviews with 512 tokens: 93.97%

Token count statistics:
Minimum tokens: 275
Maximum tokens: 512
Mean tokens: 505.8
Median tokens: 512.0

Distribution around 512 tokens:
  510 tokens: 0 reviews (0.00%)
  511 tokens: 0 reviews (0.00%)
  512 tokens: 187 reviews (93.97%)
  513 tokens: 0 reviews (0.00%)
  514 tokens: 0 reviews (0.00%)

Reviews with 512 or more tokens: 187 (93.97%)
Reviews with exactly 512 tokens: 187 (93.97%)
